In [1]:
import numpy as np
print(np.__version__)

from datetime import datetime
import torch
print(torch.__version__)

import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

from tqdm.notebook import trange

import random
import math
import time

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

1.26.4
2.9.0


In [2]:
import numpy as np
from numba import njit

SIZE = 5
EMPTY = 0
P1 = 1
P2 = -1
max_moves = 100


def build_action_map(size):
    actions = []

    for r in range(size):
        for c in range(size):
            if not (r == 0 or r == size-1 or c == 0 or c == size-1):
                continue

            if c != 0:
                actions.append((r, c, r, 0))
            if c != size - 1:
                actions.append((r, c, r, size - 1))
            if r != 0:
                actions.append((r, c, 0, c))
            if r != size - 1:
                actions.append((r, c, size - 1, c))

    return np.array(actions, dtype=np.int8)  # (A, 4)

@njit
def get_valid_moves_numba(state, action_map):
    A = action_map.shape[0]
    valid = np.zeros(A, dtype=np.uint8)

    for i in range(A):
        r, c = action_map[i, 0], action_map[i, 1]
        piece = state[r, c]

        if piece == EMPTY or piece == P1:
            valid[i] = 1

    return valid

@njit
def push_numba(state, action):
    r, c, dr, dc = action
    size = state.shape[0]

    new = state.copy()

    if r == dr:
        # row shift
        if dc == 0:
            # push from right → shift right
            for j in range(c, 0, -1):
                new[r, j] = new[r, j-1]
            new[r, 0] = P1
        else:
            # push from left → shift left
            for j in range(c, size-1):
                new[r, j] = new[r, j+1]
            new[r, size-1] = P1

    else:
        # column shift
        if dr == 0:
            for i in range(r, 0, -1):
                new[i, c] = new[i-1, c]
            new[0, c] = P1
        else:
            for i in range(r, size-1):
                new[i, c] = new[i+1, c]
            new[size-1, c] = P1

    return new

@njit
def has_line_numba(state, player):
    size = state.shape[0]

    for i in range(size):
        if np.all(state[i, :] == player):
            return True
        if np.all(state[:, i] == player):
            return True

    diag1 = True
    diag2 = True

    for i in range(size):
        if state[i, i] != player:
            diag1 = False
        if state[i, size - 1 - i] != player:
            diag2 = False

    return diag1 or diag2

def encode_state_batch(states):
    # states: (B, S, S)

    p2 = (states == P2)
    empty = (states == EMPTY)
    p1 = (states == P1)

    encoded = np.stack((p2, empty, p1), axis=1).astype(np.float32)
    # (B, 3, S, S)

    return encoded

class QuixoFast:
    def __init__(self):
        self.size = SIZE
        self.action_map = build_action_map(SIZE)   # (A, 4)
        self.action_size = self.action_map.shape[0]

        # --- NEW: fast lookup ---
        self._action_to_index = {}
        for i in range(self.action_size):
            key = tuple(self.action_map[i])
            self._action_to_index[key] = i

    def get_initial_state(self):
        return np.zeros((self.size, self.size), dtype=np.int8)

    def change_perspective(self, state, player):
        return state * player

    def get_valid_moves(self, state):
        return get_valid_moves_numba(state, self.action_map)

    def get_next_state(self, state, action, player):
        canonical = state * player
        next_state = push_numba(canonical, self.action_map[action])
        return next_state * player

    def get_value_and_terminated(self, state, player, move_count):
        if has_line_numba(state, -player):
            return -1, True
        if has_line_numba(state, player):
            return 1, True
        if move_count >= max_moves:
            return 0, True
        return 0, False
    
    def encode_action(self, r, c, dr, dc):
        return self._action_to_index[(r, c, dr, dc)]

    def decode_action(self, action):
        return self.action_map[action]
    
    def get_encoded_state(self, state):
        """
        Encodes the state into a 3-channel representation for the ResNet.
        Channels: [Opponent's Pieces, Empty Squares, Current Player's Pieces]
        """
        # Constants used: P1 = 1, P2 = -1, EMPTY = 0
        if state.ndim == 2: # Single state (5, 5)
            return np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        
        # Batch of states (Batch, 5, 5)
        # We swap axes so the output is (Batch, 3, 5, 5)
        encoded = np.stack((state == -1, state == 0, state == 1)).astype(np.float32)
        return np.swapaxes(encoded, 0, 1)

    def get_opponent(self, player):
        """Returns the other player (-1 for 1, 1 for -1)."""
        return -player

    def get_opponent_value(self, value):
        """Flips the value for the other player's perspective."""
        return -value

In [3]:
def print_board(state):
    symbols = {0: ".", 1: "X", -1: "O"}
    print()
    for r in range(state.shape[0]):
        print(" ".join(symbols[x] for x in state[r]))
    print()


def get_human_move(game, state):
    """
    Input format: r c side (or 'q' to quit)
    """
    side_map = {
        "L": lambda r, c: (r, c, r, 0),
        "R": lambda r, c: (r, c, r, game.size - 1),
        "T": lambda r, c: (r, c, 0, c),
        "B": lambda r, c: (r, c, game.size - 1, c),
    }

    valid = game.get_valid_moves(state)

    while True:
        try:
            raw_input = input("Enter move (r c side[L/R/T/B]) or 'q' to quit: ").strip().lower()
            
            # Check for quit command
            if raw_input == 'q':
                return None

            raw = raw_input.split()
            r, c = int(raw[0]), int(raw[1])
            side = raw[2].upper()

            if side not in side_map:
                raise ValueError

            move = side_map[side](r, c)

            if move not in game.action_map:
                print("Illegal geometry (not a valid push direction).")
                continue

            action = game.encode_action(*move)

            if valid[action] == 0:
                print("Illegal move (violates rules).")
                continue

            return action

        except (ValueError, IndexError):
            print("Invalid input. Format: r c side (Example: 0 0 R) or 'q'")


def play_game():
    game = QuixoFast()
    state = game.get_initial_state()
    player = 1  # X starts
    move_count = 0
    while True:
        print_board(state)
        print(f"Player {'X' if player == 1 else 'O'}")

        canonical = game.change_perspective(state, player)

        # Get the action and check if the player chose to quit
        action = get_human_move(game, canonical)
        
        if action is None:
            print("Game terminated by user.")
            break

        state = game.get_next_state(state, action, player)
        move_count += 1
        value, done = game.get_value_and_terminated(state, player, move_count)

        if done:
            print_board(state)
            if value == 1:
                print(f"Player {'X' if player == 1 else 'O'} wins!")
            else:
                print(f"Player {'X' if player == 1 else 'O'} loses!")
            break

        player = -player

# play_game()

In [ ]:
"""
PPO for Quixo – self-play training with evaluation vs. uniform-random.

Design notes
------------
* Both players share one network; states are always presented in canonical
  form (own pieces = +1, opponent = -1) so the network is player-agnostic.
* All intermediate rewards are 0; terminal reward is +1 (win), -1 (loss),
  0 (draw/timeout). Monte-Carlo returns are therefore trivially computable:
      G_i = gamma^(T-1-i) * r_terminal
* We collect complete episodes rather than fixed-length rollouts to avoid
  having to bootstrap mid-game values across the player boundary.
* Valid-move masks are stored in the replay buffer and applied both at
  rollout time and during the PPO gradient step, keeping the policy
  distribution restricted to legal actions throughout.
* Evaluation rotates PPO between player-1 and player-2 seats to control
  for first-mover advantage.

Paste (or import) the QuixoFast class above this file, then call train().
"""

import csv
import time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical


# ── device ────────────────────────────────────────────────────────────────────
device = torch.device(
    "cuda"  if torch.cuda.is_available()  else
    "mps"   if torch.backends.mps.is_available() else
    "cpu"
)

# ── hyperparameters ───────────────────────────────────────────────────────────
GAE_LAMBDA          = 0.95   # Smoothing factor for GAE
GAMMA               = 0.99   # discount; close to 1 since reward is terminal-only
CLIP_EPS            = 0.2    # PPO clipping radius
LR                  = 1e-4
VALUE_COEF          = 2      # weight of value loss relative to policy loss
ENTROPY_COEF        = 0.01    # entropy bonus; keeps exploration alive
N_EPOCHS            = 5      # passes over each collected buffer
BATCH_SIZE          = 1024
GRAD_CLIP           = 1.0    # max gradient norm
EPISODES_PER_UPDATE = 256    # self-play games collected before each PPO step
N_UPDATES           = 500
EVAL_INTERVAL       = 10     # evaluate every N updates
N_EVAL_GAMES        = 200    # games per evaluation (split evenly across sides)
VALUE_CLIP_EPS      = 0.2

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
print(f"Training started at {timestamp}")

# print the hyperparameters
print(f"GAE_LAMBDA: {GAE_LAMBDA}")
print(f"GAMMA: {GAMMA}")
print(f"CLIP_EPS: {CLIP_EPS}")
print(f"LR: {LR}")
print(f"VALUE_COEF: {VALUE_COEF}")
print(f"ENTROPY_COEF: {ENTROPY_COEF}")
print(f"N_EPOCHS: {N_EPOCHS}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"GRAD_CLIP: {GRAD_CLIP}")
print(f"EPISODES_PER_UPDATE: {EPISODES_PER_UPDATE}")
print(f"N_UPDATES: {N_UPDATES}")
print(f"EVAL_INTERVAL: {EVAL_INTERVAL}")
print(f"N_EVAL_GAMES: {N_EVAL_GAMES}")
print(f"VALUE_CLIP_EPS: {VALUE_CLIP_EPS}")

# ── network ───────────────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, ch: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )

    def forward(self, x):
        return F.relu(x + self.net(x), inplace=True)


class QuixoNet(nn.Module):
    """
    Input  : (B, 3, 5, 5)  – channels are [opponent, empty, self]
    Outputs: logits (B, A),  value (B, 1) ∈ [-1, 1]
    """
    def __init__(self, action_size: int, num_hidden: int = 64, num_res: int = 4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, num_hidden, 3, padding=1, bias=False),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU(inplace=True),
        )
        self.backbone = nn.Sequential(*[ResBlock(num_hidden) for _ in range(num_res)])

        # Policy head: conv bottleneck → flat → logits
        self.policy_head = nn.Sequential(
            nn.Conv2d(num_hidden, 2, 1, bias=False),
            nn.BatchNorm2d(2),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(2 * 5 * 5, action_size),
        )

        # Value head: conv bottleneck → MLP → tanh scalar
        self.value_head = nn.Sequential(
            nn.Conv2d(num_hidden, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(5 * 5, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.backbone(self.stem(x))
        return self.policy_head(x), self.value_head(x)


# ── helpers ───────────────────────────────────────────────────────────────────
def _encode(env, canonical_state) -> torch.Tensor:
    """Return a (3,5,5) float32 tensor on device."""
    arr = env.get_encoded_state(canonical_state)   # (3,5,5) ndarray
    return torch.as_tensor(arr, dtype=torch.float32, device=device)


def _masked_dist(logits: torch.Tensor, valid: torch.Tensor) -> Categorical:
    """Categorical distribution over legal actions only."""
    return Categorical(logits=logits + (1.0 - valid) * -1e9)


def _mc_returns(terminal_reward: float, n: int, gamma: float) -> list:
    """
    Discounted returns for a trajectory whose only non-zero reward is the last.
    G_i = gamma^(n-1-i) * terminal_reward,  computed in O(n) backwards.
    """
    G = terminal_reward
    rets = [0.0] * n
    for i in range(n - 1, -1, -1):
        rets[i] = G
        G *= gamma
    return rets

@njit
def get_valid_moves_batch_numba(states, action_map):
    """Calculates valid moves for a batch of states (B, 5, 5)."""
    B = states.shape[0]
    A = action_map.shape[0]
    valid = np.zeros((B, A), dtype=np.uint8)
    for b in range(B):
        for i in range(A):
            r, c = action_map[i, 0], action_map[i, 1]
            piece = states[b, r, c]
            if piece == 0 or piece == 1:
                valid[b, i] = 1
    return valid

def encode_gpu(canonical_states_np):
    """
    Takes a (B, 5, 5) int8 numpy array and encodes it directly on the GPU.
    Returns a (B, 3, 5, 5) float32 tensor.
    """
    t = torch.as_tensor(canonical_states_np, dtype=torch.int8, device=device)
    return torch.stack([
        (t == -1).float(),
        (t == 0).float(),
        (t == 1).float()
    ], dim=1)


# ── rollout collection ────────────────────────────────────────────────────────
def collect_episodes(env, net: QuixoNet, n_episodes: int, gamma: float, gae_lambda: float = 0.95) -> list:
    net.eval()
    buffer = []
    
    # Run 64 games simultaneously
    num_envs = 64
    
    states = np.zeros((num_envs, env.size, env.size), dtype=np.int8)
    players = np.ones(num_envs, dtype=np.int8)
    move_cts = np.zeros(num_envs, dtype=np.int32)
    trajectories = [{1: [], -1: []} for _ in range(num_envs)]
    
    episodes_collected = 0
    
    while episodes_collected < n_episodes:
        # 1. Get canonical states for all environments
        canon = states * players[:, None, None]
        
        # 2. Get valid moves and move to GPU
        valid_masks_np = get_valid_moves_batch_numba(canon, env.action_map)
        valid_t = torch.as_tensor(valid_masks_np, dtype=torch.float32, device=device)
        
        # 3. Encode states directly on GPU
        enc = encode_gpu(canon)  # Shape: (64, 3, 5, 5)
        
        # 4. Neural Network Forward Pass (Batched)
        with torch.no_grad():
            logits, values = net(enc)
            
        dist = _masked_dist(logits, valid_t)
        actions = dist.sample()
        lps = dist.log_prob(actions)
        
        # 5. Move results to CPU for game logic
        actions_np = actions.cpu().numpy()
        values_np = values.squeeze(-1).cpu().numpy()
        lps_np = lps.cpu().numpy()
        enc_cpu = enc.cpu()  # Store CPU version in buffer to save VRAM
        
        # 6. Step each environment
        for b in range(num_envs):
            if episodes_collected >= n_episodes:
                break  # Stop processing if we hit our quota
                
            # Store step data
            trajectories[b][players[b]].append({
                "state": enc_cpu[b],
                "action": actions_np[b],
                "log_prob": lps_np[b],
                "value": values_np[b],
                "valid_mask": valid_masks_np[b],
                "reward": 0.0  # Placeholder, updated on termination
            })
            
            # Push and get next state via Numba
            next_state = push_numba(canon[b], env.action_map[actions_np[b]])
            next_state = next_state * players[b]  # Revert to global perspective
            
            move_cts[b] += 1
            reward, done = env.get_value_and_terminated(next_state, players[b], move_cts[b])
            
            if done:
                # Update final rewards
                trajectories[b][players[b]][-1]["reward"] = float(reward)
                if len(trajectories[b][-players[b]]) > 0:
                    trajectories[b][-players[b]][-1]["reward"] = float(-reward)
                    
                # Calculate GAE for this specific game
                for p in (1, -1):
                    t = trajectories[b][p]
                    if not t: continue
                    
                    gae = 0
                    for i in reversed(range(len(t))):
                        if i == len(t) - 1:
                            delta = t[i]["reward"] - t[i]["value"]
                        else:
                            delta = 0 + (gamma**2) * t[i+1]["value"] - t[i]["value"]
                        
                        gae = delta + (gamma**2) * gae_lambda * gae
                        t[i]["advantage"] = gae
                        t[i]["return_"] = gae + t[i]["value"]
                        buffer.append(t[i])
                        
                episodes_collected += 1
                
                # Reset environment b
                states[b] = 0
                players[b] = 1
                move_cts[b] = 0
                trajectories[b] = {1: [], -1: []}
            else:
                states[b] = next_state
                players[b] = -players[b]
                
    return buffer
# ── PPO gradient step ─────────────────────────────────────────────────────────
def ppo_update(net, optimizer, buffer, n_epochs, batch_size, clip_eps, value_coef, ent_coef, grad_clip):
    net.train()
    
    # Pre-stack all data from the buffer
    states = torch.stack([t["state"] for t in buffer]).to(device)
    actions = torch.tensor([t["action"] for t in buffer], device=device)
    old_lps = torch.tensor([t["log_prob"] for t in buffer], dtype=torch.float32, device=device)
    returns = torch.tensor([t["return_"] for t in buffer], dtype=torch.float32, device=device)
    advantages = torch.tensor([t["advantage"] for t in buffer], dtype=torch.float32, device=device)
    valid_masks = torch.tensor(np.stack([t["valid_mask"] for t in buffer]), dtype=torch.float32, device=device)
    old_values = torch.tensor([t["value"] for t in buffer], dtype=torch.float32, device=device)

    # --- Robust Advantage Normalization ---
    # Normalize across the WHOLE buffer, not just the mini-batch
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    totals = defaultdict(float)
    n_batches = 0

    for epoch in range(n_epochs):
        perm = torch.randperm(len(buffer), device=device)
        
        for start in range(0, len(buffer) - batch_size + 1, batch_size):
            idx = perm[start:start + batch_size]
            
            logits, values = net(states[idx])
            values = values.squeeze(-1)
            
            dist = _masked_dist(logits, valid_masks[idx])
            new_lp = dist.log_prob(actions[idx])
            entropy = dist.entropy().mean()
            
            ratio = torch.exp(new_lp - old_lps[idx])
            
            # Policy loss
            surr1 = ratio * advantages[idx]
            surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages[idx]
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # Value loss (Clipped for stability)
            v_pred_clipped = old_values[idx] + torch.clamp(values - old_values[idx], -0.2, 0.2)
            value_loss = torch.max((values - returns[idx])**2, (v_pred_clipped - returns[idx])**2).mean()
            
            loss = policy_loss + value_coef * value_loss - ent_coef * entropy
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), grad_clip)
            optimizer.step()
            
            totals["policy_loss"] += policy_loss.item()
            totals["value_loss"] += value_loss.item()
            totals["entropy"] += entropy.item()
            n_batches += 1
            
    return {k: v / n_batches for k, v in totals.items()}
# ── evaluation ────────────────────────────────────────────────────────────────
def evaluate_vs_random(env, net: QuixoNet, n_games: int = 200) -> dict:
    net.eval()
    counts = {"win": 0, "draw": 0, "loss": 0}
    
    # Initialize all games
    states = np.zeros((n_games, env.size, env.size), dtype=np.int8)
    players = np.ones(n_games, dtype=np.int8)
    move_cts = np.zeros(n_games, dtype=np.int32)
    active = np.ones(n_games, dtype=bool)
    
    # Assign PPO to player 1 for first half, player -1 for second half
    ppo_sides = np.ones(n_games, dtype=np.int8)
    ppo_sides[n_games // 2:] = -1
    
    while np.any(active):
        # Only process games that haven't finished
        active_idx = np.where(active)[0]
        
        canon = states[active_idx] * players[active_idx, None, None]
        valid_masks = get_valid_moves_batch_numba(canon, env.action_map)
        
        actions = np.zeros(len(active_idx), dtype=np.int64)
        
        # Identify whose turn it is
        ppo_turn = (players[active_idx] == ppo_sides[active_idx])
        
        # 1. Get PPO moves via neural network
        if np.any(ppo_turn):
            ppo_idx = np.where(ppo_turn)[0]
            enc = encode_gpu(canon[ppo_idx])
            valid_t = torch.as_tensor(valid_masks[ppo_idx], dtype=torch.float32, device=device)
            
            with torch.no_grad():
                logits, _ = net(enc)
            dist = _masked_dist(logits, valid_t)
            actions[ppo_idx] = dist.sample().cpu().numpy()
            
        # 2. Get Random moves
        rnd_turn = ~ppo_turn
        if np.any(rnd_turn):
            rnd_idx = np.where(rnd_turn)[0]
            for i in rnd_idx:
                valid_indices = np.where(valid_masks[i])[0]
                actions[i] = np.random.choice(valid_indices)
                
        # 3. Apply actions to environments
        for i, idx in enumerate(active_idx):
            next_state = push_numba(canon[i], env.action_map[actions[i]])
            next_state = next_state * players[idx]
            move_cts[idx] += 1
            
            reward, done = env.get_value_and_terminated(next_state, players[idx], move_cts[idx])
            
            if done:
                active[idx] = False
                ppo_reward = reward if players[idx] == ppo_sides[idx] else -reward
                if ppo_reward > 0: counts["win"] += 1
                elif ppo_reward < 0: counts["loss"] += 1
                else: counts["draw"] += 1
            else:
                states[idx] = next_state
                players[idx] = -players[idx]
                
    return {k: v / n_games for k, v in counts.items()}

# ── training loop ─────────────────────────────────────────────────────────────
def train(
    n_updates           = N_UPDATES,
    episodes_per_update = EPISODES_PER_UPDATE,
    eval_interval       = EVAL_INTERVAL,
    n_eval_games        = N_EVAL_GAMES,
):
    env = QuixoFast()
    net = QuixoNet(action_size=env.action_size).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=LR)

    print(f"Device      : {device}")
    print(f"Action size : {env.action_size}")
    print(f"Parameters  : {sum(p.numel() for p in net.parameters()):,}")
    print()

    header = (
        f"{'Upd':>5}  "
        f"{'Win':>6} {'Draw':>6} {'Loss':>6}  "
        f"{'π-loss':>8} {'V-loss':>8} {'H':>7}  "
        f"{'Buf':>5}  {'s':>5}"
    )
    print(header)
    print("─" * len(header))

    log = defaultdict(list)   # accumulates everything for later plotting/analysis

    # Write CSV header and hyperparameters to disk before the main loop.
    csv_file = f"ppo_training_log_{timestamp}.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["hyperparameter", "value"])
        writer.writerow(["GAMMA", GAMMA])
        writer.writerow(["CLIP_EPS", CLIP_EPS])
        writer.writerow(["LR", LR])
        writer.writerow(["VALUE_COEF", VALUE_COEF])
        writer.writerow(["ENTROPY_COEF", ENTROPY_COEF])
        writer.writerow(["N_EPOCHS", N_EPOCHS])
        writer.writerow(["BATCH_SIZE", BATCH_SIZE])
        writer.writerow(["GRAD_CLIP", GRAD_CLIP])
        writer.writerow(["EPISODES_PER_UPDATE", EPISODES_PER_UPDATE])
        writer.writerow(["N_UPDATES", N_UPDATES])
        writer.writerow(["EVAL_INTERVAL", EVAL_INTERVAL])
        writer.writerow(["N_EVAL_GAMES", N_EVAL_GAMES])
        writer.writerow([])
        writer.writerow([
            "update", "win_rate", "draw_rate", "loss_rate",
            "policy_loss", "value_loss", "entropy", "buffer_size", "seconds",
        ])

    for upd in range(1, n_updates + 1):
        t0 = time.perf_counter()

        buffer  = collect_episodes(env, net, episodes_per_update, GAMMA)
        metrics = ppo_update(
            net, opt, buffer,
            N_EPOCHS, BATCH_SIZE, CLIP_EPS, VALUE_COEF, ENTROPY_COEF, GRAD_CLIP,
        )

        dt = time.perf_counter() - t0

        log["update"].append(upd)
        log["policy_loss"].append(metrics["policy_loss"])
        log["value_loss"].append(metrics["value_loss"])
        log["entropy"].append(metrics["entropy"])
        log["buffer_size"].append(len(buffer))

        # Evaluate and log every update (or use run frequency via eval_interval)
        if upd % eval_interval == 0:
            rates = evaluate_vs_random(env, net, n_eval_games)
        else:
            rates = evaluate_vs_random(env, net, max(10, n_eval_games // 10))

        log["eval_update"].append(upd)
        log["win_rate"].append(rates["win"])
        log["draw_rate"].append(rates["draw"])
        log["loss_rate"].append(rates["loss"])

        eval_cols = f"{rates['win']:6.1%} {rates['draw']:6.1%} {rates['loss']:6.1%}"

        # Persist log to file after each update
        log_row = [
            upd,
            rates["win"],
            rates["draw"],
            rates["loss"],
            metrics["policy_loss"],
            metrics["value_loss"],
            metrics["entropy"],
            len(buffer),
            dt,
        ]
        with open(f"ppo_training_log_{timestamp}.csv", "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(log_row)

        print(
            f"{upd:5d}  "
            f"{eval_cols}  "
            f"{metrics['policy_loss']:8.4f} {metrics['value_loss']:8.4f} "
            f"{metrics['entropy']:7.4f}  "
            f"{len(buffer):5d}  {dt:5.1f}"
        )

    return net, dict(log)

if __name__ == "__main__":
    net, log = train()

Training started at 2026_04_01_20_00_56
GAE_LAMBDA: 0.95
GAMMA: 0.99
CLIP_EPS: 0.2
LR: 0.0001
VALUE_COEF: 2
ENTROPY_COEF: 0.01
N_EPOCHS: 5
BATCH_SIZE: 1024
GRAD_CLIP: 1.0
EPISODES_PER_UPDATE: 256
N_UPDATES: 500
EVAL_INTERVAL: 10
N_EVAL_GAMES: 200
VALUE_CLIP_EPS: 0.2
Device      : mps
Action size : 44
Parameters  : 301,963

  Upd     Win   Draw   Loss    π-loss   V-loss       H    Buf      s
────────────────────────────────────────────────────────────────────
    1   55.0%   0.0%  45.0%   -0.0081   0.2697  3.3016  11342    7.0
